# Practice #4. "ARIMA models and advanced techniques"

Residual modelling, ADF *and* KPSS, ARIMA order selection, diagnostics.

**From Practice 3:** the ADF test and differencing. The `d` in ARIMA(p, d, q) is
exactly the number of differences `make_stationary` applied.

Fill in the cells tagged `graded`, keeping every name and signature exactly as
given — they are graded automatically.

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

def find_data_dir():
    """The repo's data/ directory, wherever the kernel happens to start."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "airline-passengers.csv").exists():
            return folder / "data"
    raise FileNotFoundError("data/ not found - run this notebook inside the repo")


DATA_DIR = globals().get("DATA_DIR", find_data_dir())

## 0. Data

`airline-passengers.csv` — monthly airline passengers, 1949-1960. Rising trend
*and* seasonality whose amplitude grows with the level: the standard ARIMA
example.

Open the file and find the column names yourself.

In [ ]:
def load_series(path, time_col, value_col):
    """Read a CSV into a float Series named "y" on a DatetimeIndex."""
    # TODO: same contract as Practices 1-3. Column names differ per file —
    # open the csv.
    raise NotImplementedError

In [ ]:
series = load_series(DATA_DIR / "airline-passengers.csv", "Month", "Passengers")
train, test = series.iloc[:-24], series.iloc[-24:]
print(f"{len(series)} points; train {len(train)}, test {len(test)}")

plt.figure(figsize=(20, 5))
plt.plot(train, label="train")
plt.plot(test, label="test")
plt.legend();

## 1. Modelling the residual error

An AR model on a trending series leaves structure in its residuals. If those
residuals are still autocorrelated, a second model can predict them, and adding
that prediction back corrects the first forecast.

That is what the MA term in ARMA formalises.

In [ ]:
def residual_corrected_forecast(train, index, lags, resid_lags):
    """AR forecast plus a second AR model fitted to its own residuals.

    1. AutoReg(lags) on `train` -> forecast over `index`
    2. take that fit's in-sample residuals
    3. AutoReg(resid_lags) on the residuals -> forecast over `index`
    4. return the sum, as a Series on `index`
    """
    # TODO
    raise NotImplementedError

In [ ]:
plain = AutoReg(np.asarray(train), lags=12, old_names=False).fit()
plain_forecast = pd.Series(np.asarray(plain.forecast(steps=len(test))),
                           index=test.index)
corrected = residual_corrected_forecast(train, test.index, lags=12, resid_lags=6)

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(predicted)) ** 2)))

print(f"AR(12) alone      RMSE {rmse(test, plain_forecast):.3f}")
print(f"AR(12) + residual RMSE {rmse(test, corrected):.3f}")

plt.figure(figsize=(20, 5))
plt.plot(train.iloc[-48:], label="train")
plt.plot(test, label="test")
plt.plot(plain_forecast, "--", label="AR(12)")
plt.plot(corrected, label="AR(12) + residual model")
plt.legend();

**Question.** Did the correction help? If the first model's residuals
were pure white noise, what would the second model contribute, and why?

## 2. Stationarity: two tests, opposite nulls

Trusting one test alone is how people talk themselves into the wrong `d`.

| | null hypothesis | small p means |
|---|---|---|
| ADF | has a unit root | stationary |
| KPSS | is stationary | non-stationary |

Agreement gives you an answer. Disagreement is also a result — usually
trend-stationary rather than difference-stationary. Report it, don't pick a side.

In [ ]:
def check_stationarity(series, significance=0.05):
    """Run ADF and KPSS, report both plus a combined verdict.

    Returns {"adf_p", "kpss_p", "adf_stationary", "kpss_stationary",
    "verdict"}, where verdict is "stationary", "non-stationary" or
    "inconclusive" (the two tests disagree).

    Opposite nulls: small ADF p => stationary, small KPSS p => NOT stationary.
    """
    # TODO
    raise NotImplementedError

In [ ]:
for label, candidate in (("raw", train),
                         ("log", np.log(train)),
                         ("log + 1 diff", np.log(train).diff().dropna()),
                         ("log + 1 + 12 diff",
                          np.log(train).diff().diff(12).dropna())):
    result = check_stationarity(candidate)
    print(f"{label:>18}: ADF p={result['adf_p']:.4f}  "
          f"KPSS p={result['kpss_p']:.4f}  ->  {result['verdict']}")

**Question.** Raw and logged give almost the same verdict. So what does
the log transform actually fix here, and which plot shows it?

### 2.1 Reading p, d and q off the plots

- PACF cuts off after lag p → AR(p)
- ACF cuts off after lag q → MA(q)
- d = differences needed to reach stationarity

In [ ]:
stationary = np.log(train).diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(20, 4))
plot_acf(stationary, lags=36, ax=axes[0], title="ACF — log, 1 difference")
plot_pacf(stationary, lags=36, ax=axes[1], title="PACF — log, 1 difference")
plt.tight_layout();

## 3. Automatic order selection

The plots give you a candidate; an AIC grid search gives a second opinion. When
they disagree, prefer the simpler model that passes the section 4 diagnostics —
a low AIC with autocorrelated residuals is not worth having.

In [ ]:
def select_order(series, max_p=3, max_d=2, max_q=3):
    """Return the (p, d, q) with the lowest AIC.

    Some orders are not estimable on a given series and raise. Skip those —
    one bad fit must not abort the search.
    """
    # TODO: itertools.product over the three ranges, ARIMA(...).fit() in a
    # try/except, track the best .aic.
    raise NotImplementedError

In [ ]:
order = select_order(train, max_p=3, max_d=2, max_q=3)
print(f"best order by AIC: {order}")

## 4. Forecasting and diagnostics

**The drift trap.** Once `d >= 1`, statsmodels' `ARIMA` adds **no constant term
by default**. The forecast is then a flat line at the last level: it will not
extrapolate a trend, and can lose to naive. On a synthetic series of slope 0.5:

| model | RMSE |
|---|---|
| `ARIMA(order=(1,1,0))` (default) | 6.47 |
| `ARIMA(order=(1,1,0), trend="t")` | 0.75 |
| naive | 6.43 |

Pass `trend="t"` when you think the series drifts — and say why you think so.

In [ ]:
def arima_forecast(train, index, order, trend=None):
    """Fit ARIMA(order) on `train`, forecast over `index`."""
    # TODO: pass `trend` straight through to ARIMA, and leave it out when the
    # caller passes None. See the drift trap above for why that matters.
    raise NotImplementedError


def ljung_box_pvalues(residuals, lags=10):
    """Ljung-Box p-values for lags 1..lags, as a Series indexed by lag.

    Null is "residuals are independent", so p ABOVE 0.05 is the good outcome:
    nothing left to model.
    """
    # TODO: acorr_ljungbox returns a DataFrame with an "lb_pvalue" column.
    raise NotImplementedError

In [ ]:
forecast = arima_forecast(train, test.index, order)
drifted = arima_forecast(train, test.index, order, trend="t")
print(f"ARIMA{order}            RMSE {rmse(test, forecast):.3f}")
print(f"ARIMA{order} trend='t'  RMSE {rmse(test, drifted):.3f}")
print(f"naive     RMSE {rmse(test, np.full(len(test), train.iloc[-1])):.3f}")

plt.figure(figsize=(20, 5))
plt.plot(train.iloc[-48:], label="train")
plt.plot(test, label="test")
plt.plot(forecast, label=f"ARIMA{order}")
plt.plot(drifted, "--", label=f"ARIMA{order} trend='t'")
plt.legend();

### 4.1 Residual diagnostics

A model you can trust leaves residuals that look like white noise: no
autocorrelation, roughly constant variance, roughly symmetric. Ljung-Box checks
the first — and here a **large** p-value is the good one.

In [ ]:
fitted = ARIMA(np.asarray(train), order=order).fit()
residuals = pd.Series(fitted.resid, index=train.index)
p_values = ljung_box_pvalues(residuals, lags=12)

print(p_values.round(4).to_string())
worst = p_values.min()
print(f"\nsmallest p-value {worst:.4f} -> "
      f"{'residuals look like white noise' if worst > 0.05 else 'structure remains'}")

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
axes[0].plot(residuals); axes[0].set_title("residuals")
axes[1].hist(residuals, bins=25); axes[1].set_title("distribution")
plot_acf(residuals, lags=24, ax=axes[2], title="ACF of residuals")
plt.tight_layout();

**Questions.**
1. Does your order pass Ljung-Box at every lag? If not, which lag fails — and
   what seasonal period is that?
2. `select_order` never tries seasonal terms. From the ACF, what would `SARIMAX`
   with a seasonal order add here?
3. Compare against the Holt-Winters forecast from Practice 2 over the same 24
   months. Which wins, and by enough to justify the extra machinery?

In [ ]:
# your code here — free exploration, not graded